In [2]:
# cell 1
import numpy as np, astropy
from astropy.table import Table
print(np.__version__, astropy.__version__)

2.5.4 8.0.2


## IPL4_agesJT.fits

In [3]:
# cell 2: the small file only, read fully into memory (no memmap)
from pathlib import Path
DATA_DIR = Path.cwd().parent / "data"
a = Table.read(DATA_DIR / "IPL42_agesJT.fits", memmap=False)
print(len(a), a.colnames)

921442 ['sdss_id', 'AgeJTsketchy']


In [4]:
# cell 3: the big file, no memmap (about 1 GB in memory, fine in a fresh kernel)
SAMPLE_FILE = DATA_DIR / "final_sample_zsun0_zgrbits45.fits"
f = Table.read(SAMPLE_FILE, memmap=False)
print(len(f))
ids_f = np.asarray(np.ma.filled(f["sdss_id"], -1)).astype(np.int64)
del f

515270


In [5]:
# cell A
print(a.colnames)
KEY_AGES, AGE_COL = "sdss_id", "AgeJTsketchy"      # edit KEY_AGES if the catalog uses another ID
ids_a = np.array(a[KEY_AGES]).astype(np.int64)
print(ids_a.dtype, len(ids_a), ids_a[:5])

['sdss_id', 'AgeJTsketchy']
int64 921442 [ 69834792 117290797  54400284  55478193  71380512]


In [6]:
# cell B
age = np.array(a[AGE_COL]).astype(float)
has_age = np.isfinite(age) & (age > 0)
print(has_age.sum(), "catalog rows with a finite positive age")

920674 catalog rows with a finite positive age


In [7]:
# 1
print(type(ids_f), ids_f.dtype, ids_f.shape, ids_f[:5])

<class 'numpy.ndarray'> int64 (515270,) [117290797  54400284  55478193  54526458  54402433]


In [8]:
# 2
print((ids_f > 0).sum())

515270


In [9]:
# 3
cat_ids = set(ids_a[ids_a > 0].tolist())
print(len(cat_ids))

852131


In [10]:
# 4
ids_f_list = ids_f.tolist()
print(len(ids_f_list))

515270


In [11]:
# 5: only catalog rows with a finite positive age
cat_ids = set(ids_a[has_age].tolist())
print(len(cat_ids), "unique sdss_id with an age")

n_match = 0
for x in ids_f_list:
    if x in cat_ids:
        n_match += 1
print(n_match, "of", len(ids_f_list), "final-sample stars have an age")

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [12]:
print(age.shape, a[AGE_COL].dtype)
print(age[:5])

(921442, 1) >f8
[[4.40873639]
 [3.68056842]
 [3.43072543]
 [3.83612284]
 [8.37657777]]


In [13]:
age = np.array(a[AGE_COL]).astype(float).ravel()
has_age = np.isfinite(age) & (age > 0)
print(has_age.sum(), "catalog rows with an age")

cat_ids = set(ids_a[has_age].tolist())
print(len(cat_ids), "unique sdss_id with an age")

n_match = sum(1 for x in ids_f_list if x in cat_ids)
print(n_match, "of", len(ids_f_list), "final-sample stars have an age")

920674 catalog rows with an age
851403 unique sdss_id with an age
458150 of 515270 final-sample stars have an age


In [14]:
ids_c  = ids_a[has_age]
ages_c = age[has_age]

u, inv = np.unique(ids_c, return_inverse=True)
cnt = np.bincount(inv)
mean_age = np.bincount(inv, weights=ages_c) / cnt

# how much do duplicate rows disagree?
mx = np.full(len(u), -np.inf); np.maximum.at(mx, inv, ages_c)
mn = np.full(len(u),  np.inf); np.minimum.at(mn, inv, ages_c)
dup = cnt > 1
print(dup.sum(), "stars with duplicate rows; median max-min spread:",
      np.median((mx - mn)[dup]))

# attach to the final sample, in its row order
pos = np.clip(np.searchsorted(u, ids_f), 0, len(u) - 1)
found = u[pos] == ids_f
age_f = np.where(found, mean_age[pos], np.nan)
print(found.sum(), "matched")   # should print 458150

np.savez(DATA_DIR / "final_sample_ages.npz", sdss_id=ids_f, age=age_f)

53785 stars with duplicate rows; median max-min spread: 0.45655648185998876
458150 matched


In [15]:
a_ok = age_f[np.isfinite(age_f)]
print(len(a_ok), "stars")
print("mean   :", a_ok.mean())
print("median :", np.median(a_ok))
print("16-84% :", np.percentile(a_ok, [16, 84]))

458150 stars
mean   : 4.475428939037117
median : 4.274583824682905
16-84% : [2.272849   6.85578389]


In [16]:
print(f"{100 * found.sum() / len(ids_f):.1f}% of final-sample stars have an age "
      f"({found.sum()} of {len(ids_f)})")

88.9% of final-sample stars have an age (458150 of 515270)


In [17]:
a_ok = age_f[np.isfinite(age_f)]
frac_old = (a_ok >= 3.0).mean()
print(f"{100 * frac_old:.1f}% of stars with an IPL42 age are 3 Gyr or older "
      f"({int((a_ok >= 3.0).sum()):,} of {len(a_ok):,})")
print(f"{100 * (a_ok < 3.0).mean():.1f}% are younger than 3 Gyr")

73.5% of stars with an IPL42 age are 3 Gyr or older (336,739 of 458,150)
26.5% are younger than 3 Gyr


## Starflow_summary

In [17]:
AGES2_FILE = DATA_DIR / "StarFlow_summary_v1_0_0.fits"
b = Table.read(AGES2_FILE, memmap=False)
print(len(b), "rows")
print(b.colnames)

print("age-like:", [c for c in b.colnames if "age" in c.lower()])
print("id-like :", [c for c in b.colnames if "id" in c.lower() or "gaia" in c.lower()])

1059521 rows
['sdss_id', 'sdss4_apogee_id', 'age', 'e_p_age', 'e_n_age', 'mass', 'e_p_mass', 'e_n_mass', 'training_density', 'BITMASK']
age-like: ['age', 'e_p_age', 'e_n_age']
id-like : ['sdss_id', 'sdss4_apogee_id']


In [18]:
b = Table.read(DATA_DIR / "StarFlow_summary_v1_0_0.fits", memmap=False)
print(b.colnames)

AGE2 = "age"
cands = [c for c in b.colnames if c.lower() in ("sdss_id", "gaia_source_id", "source_id", "gaia_dr3_source_id")]
print("ID candidates:", cands)
KEY2 = cands[0]
print("matching on", KEY2)

ids_b = np.asarray(np.ma.filled(b[KEY2], -1)).astype(np.int64)
age_b = np.asarray(np.ma.filled(b[AGE2].astype(float), np.nan)).ravel()
ok = np.isfinite(age_b) & (age_b > 0) & (ids_b > 0)
print(ok.sum(), "of", len(b), "rows with a usable age")

u2, inv2 = np.unique(ids_b[ok], return_inverse=True)
mean2 = np.bincount(inv2, weights=age_b[ok]) / np.bincount(inv2)

pos2 = np.clip(np.searchsorted(u2, ids_f), 0, len(u2) - 1)
found2 = u2[pos2] == ids_f
age_f2 = np.where(found2, mean2[pos2], np.nan)
print(f"{100*found2.sum()/len(ids_f):.1f}% of final-sample stars have an age "
      f"({found2.sum()} of {len(ids_f)})")

['sdss_id', 'sdss4_apogee_id', 'age', 'e_p_age', 'e_n_age', 'mass', 'e_p_mass', 'e_n_mass', 'training_density', 'BITMASK']
ID candidates: ['sdss_id']
matching on sdss_id
952170 of 1059521 rows with a usable age
43.4% of final-sample stars have an age (223411 of 515270)


In [19]:
both = np.isfinite(age_f) & np.isfinite(age_f2)
print(both.sum(), "stars have an age in both catalogs")
print("in either:", (np.isfinite(age_f) | np.isfinite(age_f2)).sum())
d = age_f2[both] - age_f[both]
print("StarFlow minus IPL42: median", np.median(d), " scatter (std)", d.std())

206270 stars have an age in both catalogs
in either: 475291
StarFlow minus IPL42: median -0.4891961320202083  scatter (std) 1.3208388715115724


In [20]:
a2 = age_f2[np.isfinite(age_f2)]
print(len(a2), "stars")
print("mean   :", a2.mean())
print("median :", np.median(a2))
print("16-84% :", np.percentile(a2, [16, 84]))

223411 stars
mean   : 4.4664031952249665
median : 4.348697394789579
16-84% : [2.35671343 6.59318637]
